# Verify random-projection CNN sensor processing

Construct a training-free sensor processor and check its feature geometry, projection, output shape, repeatability, and batch-one latency. The pretrained backbone weights may be downloaded by torchvision the first time a model is used.

In [ ]:
import pathlib
import sys
import time
sys.path.append("..")

import numpy as np
import torch
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.sp_factory import create_sp

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
experiment = "sensorprocessing_random_projection_cnn"
run = "resnet50_rademacher_128"
timing_repetitions = 10

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment(experiment)
if results_path:
    results_path = pathlib.Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(
    experiment, run, creation_style=creation_style
)
sp = create_sp(exp)

## Model and projection

In [ ]:
model = sp.enc
summary = {
    "model": exp["model"],
    "image_size": exp["image_size"],
    "feature_pool_size": exp["feature_pool_size"],
    "feature_size": model.feature_size,
    "latent_size": model.latent_size,
    "projection_shape": tuple(model.projection.shape),
    "projection_storage_mib": (
        model.projection.nelement() * model.projection.element_size() / 2**20
    ),
    "trainable_parameters": sum(
        parameter.numel() for parameter in model.parameters()
        if parameter.requires_grad
    ),
}
summary

## Repeatability and runtime interface

In [ ]:
generator = torch.Generator(device="cpu")
generator.manual_seed(73017)
images = torch.rand(
    2, 3, *exp["image_size"], generator=generator
).to(Config().runtime["device"])

with torch.inference_mode():
    first = model.encode(images)
    second = model.encode(images)

if first.shape != (2, exp["latent_size"]):
    raise ValueError(f"Unexpected latent shape: {tuple(first.shape)}")
if not torch.isfinite(first).all():
    raise FloatingPointError("Latent contains non-finite values")
if not torch.equal(first, second):
    raise RuntimeError("Repeated encoding produced different latents")

runtime_latent = sp.process(images[:1])
if runtime_latent.shape != (exp["latent_size"],):
    raise ValueError(f"Unexpected runtime shape: {runtime_latent.shape}")
if not np.all(np.isfinite(runtime_latent)):
    raise FloatingPointError("Runtime latent contains non-finite values")

{
    "repeatable": True,
    "latent_mean": float(first.mean()),
    "latent_std": float(first.std()),
    "latent_norms": first.norm(dim=1).cpu().tolist(),
}

## Batch-one latency

In [ ]:
with torch.inference_mode():
    model.encode(images[:1])
    started = time.perf_counter()
    for _ in range(timing_repetitions):
        model.encode(images[:1])
    elapsed = time.perf_counter() - started

{
    "timing_repetitions": timing_repetitions,
    "mean_batch_one_latency_ms": 1000 * elapsed / timing_repetitions,
}